# Đề bài về nhà

## Yêu cầu

- Tự viết code cho mô hình Linear Regression theo công thức đã được dạy trong buổi lý thuyết trên lớp.
- Tự viết hàm dự đoán.
- Huấn luyện cả mô hình của thư viện và mô hình mình tự viết.
- In ra các trọng số: w0, w1, w2, ..., wn của cả 2 mô hình đã huấn luyện để quan sát và so sánh.
- Dự đoán dữ liệu tập test bằng cả 2 mô hình (mô hình thư viện thì dùng hàm predict() của thư viện, mô hình tự viết dùng hàm dự đoán tự viết), in ra kết quả bằng Dataframe như trong bài thực hành trên lớp.
- Tính RMSE trên tập test cho cả 2 mô hình và so sánh.

## Dữ liệu

Tập dữ liệu giá nhà ở Boston đã có sẵn trên sklearn, dữ liệu đã được chuẩn hóa và chia thành tập train, tập test

In [38]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import math

from sklearn import datasets, linear_model
from sklearn.metrics import mean_squared_error, r2_score

# Đọc dữ liệu

Dữ liệu về giá nhà ở Boston được hỗ trợ bởi sklearn, đọc dữ liệu thông qua hàm `datasets.load_boston()`

Xem thêm các bộ dữ liệu khác tại https://scikit-learn.org/stable/datasets/index.html#toy-datasets.

Dữ liệu được chia thành các thành phần data và target như tập diabetes. Dữ liệu cũng đã được chuẩn hóa, chỉ cần gọi ra và huấn luyện

In [39]:
import pandas as pd
import numpy as np

# Fetch the Boston housing dataset from the original source
data_url = "http://lib.stat.cmu.edu/datasets/boston"
raw_df = pd.read_csv(data_url, sep="\s+", skiprows=22, header=None)
data = np.hstack([raw_df.values[::2, :], raw_df.values[1::2, :2]])
target = raw_df.values[1::2, 2]

# Create a container class to mimic the scikit-learn bunch object for compatibility with the rest of your code
class Bunch:
    def __init__(self, **kwargs):
        self.__dict__.update(kwargs)

dataset = Bunch(data=data, target=target)

print("Số chiều dữ liệu input: ", dataset.data.shape)
print("Số chiều dữ liệu target: ", dataset.target.shape)
print()

print("5 mẫu dữ liệu đầu tiên:")
print("input: ", dataset.data[:5])
print("target: ", dataset.target[:5])

<>:6: SyntaxWarning: invalid escape sequence '\s'
<>:6: SyntaxWarning: invalid escape sequence '\s'
/tmp/ipykernel_7285/726840070.py:6: SyntaxWarning: invalid escape sequence '\s'
  raw_df = pd.read_csv(data_url, sep="\s+", skiprows=22, header=None)


Số chiều dữ liệu input:  (506, 13)
Số chiều dữ liệu target:  (506,)

5 mẫu dữ liệu đầu tiên:
input:  [[6.3200e-03 1.8000e+01 2.3100e+00 0.0000e+00 5.3800e-01 6.5750e+00
  6.5200e+01 4.0900e+00 1.0000e+00 2.9600e+02 1.5300e+01 3.9690e+02
  4.9800e+00]
 [2.7310e-02 0.0000e+00 7.0700e+00 0.0000e+00 4.6900e-01 6.4210e+00
  7.8900e+01 4.9671e+00 2.0000e+00 2.4200e+02 1.7800e+01 3.9690e+02
  9.1400e+00]
 [2.7290e-02 0.0000e+00 7.0700e+00 0.0000e+00 4.6900e-01 7.1850e+00
  6.1100e+01 4.9671e+00 2.0000e+00 2.4200e+02 1.7800e+01 3.9283e+02
  4.0300e+00]
 [3.2370e-02 0.0000e+00 2.1800e+00 0.0000e+00 4.5800e-01 6.9980e+00
  4.5800e+01 6.0622e+00 3.0000e+00 2.2200e+02 1.8700e+01 3.9463e+02
  2.9400e+00]
 [6.9050e-02 0.0000e+00 2.1800e+00 0.0000e+00 4.5800e-01 7.1470e+00
  5.4200e+01 6.0622e+00 3.0000e+00 2.2200e+02 1.8700e+01 3.9690e+02
  5.3300e+00]]
target:  [24.  21.6 34.7 33.4 36.2]


**Chia dữ liệu làm 2 phần training 362 mẫu và testing 80 mẫu**

In [40]:
# cat nho du lieu, lay 1 phan cho qua trinh thu nghiem,
# chia train test cac mau du lieu
# dataset_X = dataset.data[:, np.newaxis, 2]
dataset_X = dataset.data

dataset_X_train = dataset_X[:404]
dataset_y_train = dataset.target[:404]

dataset_X_test = dataset_X[405:]
dataset_y_test = dataset.target[405:]

# Xây dựng mô hình

## Xây dựng mô hình bằng thư viện

In [41]:
from sklearn.linear_model import LinearRegression

regr = LinearRegression()
regr.fit(dataset_X_train, dataset_y_train)

LinearRegression()

## Xây dựng mô hình Linear Regression tự viết

In [46]:
class MyLinearRegression:
    def __init__(self, learning_rate=0.01, tol=1e-6, max_iter=100000):
        self.lr = learning_rate
        self.tol = tol
        self.max_iter = max_iter
        self.w = None

    def fit(self, X, y):
        X_b = np.c_[np.ones((X.shape[0], 1)), X]
        m, n = X_b.shape

        self.w = np.zeros(n)
        pre_mse = float('inf')

        for i in range(self.max_iter):
            predictions = X_b.dot(self.w)
            errors = predictions - y
            gradients = (2/m) * X_b.T.dot(errors)
            self.w = self.w - self.lr * gradients

            current_mse = np.mean(errors**2)

            if abs(current_mse - pre_mse) < self.tol:
                print(f"Hội tụ tại vòng lặp thứ {i}")
                break
            pre_mse = current_mse

    def predict(self, X):
        # Fix: Remove self.scaler since it isn't defined
        X_b = np.c_[np.ones((X.shape[0], 1)), X]
        return X_b.dot(self.w)

# Khởi tạo và huấn luyện
# Lưu ý: Do dữ liệu Boston chưa chuẩn hóa, cần learning_rate rất nhỏ
my_regr = MyLinearRegression(learning_rate=0.00000001, tol=1e-7)
my_regr.fit(dataset_X_train, dataset_y_train)

## Hàm test mô hình tự viết

# Huấn luyện mô hình

## Huấn luyện mô hình của thư viện

## Training mô hình bằng Linear regression tự viết

# Dự đoán các mẫu dữ liệu

## Dự đoán các mẫu dữ liệu theo mô hình của thư viện

## Dự đoán các mẫu dữ liệu tính theo linear regression tự viết

## Đánh giá mô hình linear regression của thư viện

## Đánh giá mô hình linear regression tự viết

In [47]:
# 1. In trọng số của cả 2 mô hình để so sánh
print("--- TRỌNG SỐ MÔ HÌNH ---")
print("Thư viện (w0 + coefficients):", [regr.intercept_] + list(regr.coef_))
print("Tự viết (w):", list(my_regr.w))

# 2. Dự đoán trên tập test
y_pred_sklearn = regr.predict(dataset_X_test)
y_pred_my = my_regr.predict(dataset_X_test)

# 3. Tạo Dataframe so sánh kết quả (10 mẫu đầu tiên)
df_result = pd.DataFrame({
    'Giá thực tế': dataset_y_test,
    'Dự đoán (Thư viện)': y_pred_sklearn,
    'Dự đoán (Tự viết)': y_pred_my
})
print("\n--- SO SÁNH KẾT QUẢ DỰ ĐOÁN ---")
display(df_result.head(10))

# 4. Tính RMSE và so sánh
rmse_sklearn = np.sqrt(mean_squared_error(dataset_y_test, y_pred_sklearn))
rmse_my = np.sqrt(mean_squared_error(dataset_y_test, y_pred_my))

print(f"\n--- ĐÁNH GIÁ RMSE ---")
print(f"RMSE Thư viện: {rmse_sklearn:.4f}")
print(f"RMSE Tự viết: {rmse_my:.4f}")

--- TRỌNG SỐ MÔ HÌNH ---
Thư viện (w0 + coefficients): [np.float64(30.077166922901817), np.float64(-0.2021352971859083), np.float64(0.04412763409501802), np.float64(0.05267393641330281), np.float64(1.884743148675411), np.float64(-14.928148681905576), np.float64(4.760386731823949), np.float64(0.002887345269762336), np.float64(-1.3002527813545937), np.float64(0.4616619529949539), np.float64(-0.015543467315894559), np.float64(-0.8116323690234529), np.float64(-0.001971744330205754), np.float64(-0.5322734312105184)]
Tự viết (w): [np.float64(0.000629723253853344), np.float64(-0.016959433754870395), np.float64(0.06155302525216137), np.float64(-0.010134611788576438), np.float64(0.0009945322801243424), np.float64(0.00024778714849601064), np.float64(0.011919101577061934), np.float64(0.00022560643072950149), np.float64(-0.0011637136210712966), np.float64(-0.00035737226623815403), np.float64(-0.010946067066385033), np.float64(-0.004050561828457338), np.float64(0.07293188993549049), np.float64(-0.0

,Giá thực tế,Dự đoán (Thư viện),Dự đoán (Tự viết)
0,5.0,3.787057,18.115107
1,11.9,6.640550,17.800874
2,27.9,21.312765,15.837315
3,17.2,15.412714,13.812124
4,27.5,23.652298,4.226393
5,15.0,16.585687,-8.738593
6,17.2,22.239823,-6.377893
7,17.9,4.597394,-7.703391
8,16.3,12.316953,6.252849
9,7.0,-4.441254,-3.975241



--- ĐÁNH GIÁ RMSE ---
RMSE Thư viện: 5.7495
RMSE Tự viết: 11.2555
